In [1]:
import os

if not os.path.exists("COVID-ML"):
    !git clone https://github.com/vinimicius/COVID-ML.git
%cd COVID-ML

Cloning into 'COVID-ML'...
remote: Enumerating objects: 96, done.
remote: Counting objects: 100% (87/87), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 96 (delta 42), reused 68 (delta 26), pack-reused 9 (from 2)
Receiving objects: 100% (96/96), 70.97 MiB | 22.82 MiB/s, done.
Resolving deltas: 100% (42/42), done.
/content/COVID-ML


In [2]:
!git checkout vini-tests

Branch 'vini-tests' set up to track remote branch 'vini-tests' from 'origin'.
Switched to a new branch 'vini-tests'


In [3]:
import os
import pandas as pd
from data_utils import run_pipeline

# Caminho do dado tratado
READY_DATA_PATH = 'data/data_ready.csv'

if os.path.exists(READY_DATA_PATH):
    print(f"✅ Carregando dados já tratados de: {READY_DATA_PATH}")
    df = pd.read_csv(READY_DATA_PATH)
else:
    print("🚀 Dados tratados não encontrados. Iniciando pipeline completa...")
    df = run_pipeline(export=True)

df.head()

🚀 Dados tratados não encontrados. Iniciando pipeline completa...
🚀 Iniciando Pipeline no diretório: data
Iniciando extração de data/data.zip...
✅ Sucesso! Dados extraídos em: data/data_raw.csv
📖 Carregando dados de: data/data_raw.csv...
📊 Dataset carregado com sucesso: 6,734,344 linhas e 20 colunas.
🧹 Iniciando limpeza dos dados...
   - 4 colunas desnecessárias removidas.
   - Valores 'IGNORADO' convertidos para 'NÃO'.
   - 217910 linhas removidas por idade inválida (<0 ou >120).

--- Relatório de Limpeza ---
✅ Dimensões finais: 6,516,433 linhas e 16 colunas.
📉 Retenção de dados: 96.76% do volume original.
🧬 Aplicando One-Hot Encoding em 'cs_sexo'...
📏 Aplicando MinMaxScaler na idade (Max original: 120.0)...
⚠️ Alerta: Colunas com texto detectadas: ['asma', 'cardiopatia', 'diabetes', 'doenca_hematologica', 'doenca_hepatica', 'doenca_neurologica', 'doenca_renal', 'imunodepressao', 'obesidade', 'outros_fatores_de_risco', 'pneumopatia', 'puerpera', 'sindrome_de_down']
   - Coluna 'asma' c

,sexo_FEMININO,sexo_IGNORADO,sexo_INDEFINIDO,sexo_MASCULINO,idade,obito,asma,cardiopatia,diabetes,doenca_hematologica,doenca_hepatica,doenca_neurologica,doenca_renal,imunodepressao,obesidade,outros_fatores_de_risco,pneumopatia,puerpera,sindrome_de_down,idade_raw
0,0,0,0,1,0.575000,0,0,0,0,0,0,0,0,0,0,0,0,0,0,69
1,1,0,0,0,0.500000,0,0,0,0,0,0,0,0,0,0,0,0,0,0,60
2,0,0,0,1,0.483333,0,0,0,0,0,0,0,0,0,0,0,0,0,0,58
3,1,0,0,0,0.375000,0,0,0,0,0,0,0,0,0,0,0,0,0,0,45
4,1,0,0,0,0.350000,0,0,0,0,0,0,0,0,0,0,0,0,0,0,42


In [4]:
# Definição clara para o leitor do notebook
#AGE_GROUPS = {
#    'jovem': (0, 30),
#    'jovem-adulto': (20, 40),
#    'adulto': (40, 60),
#    'senior': (60, 80),
#    'idoso': (80, 120)
#}

In [5]:
# Definição clara para o leitor do notebook
AGE_GROUPS = {'idoso': (80, 120)
}

In [6]:
# Dicionário que servirá como nosso banco de dados de resultados
all_reports = {}

# Definimos o nosso grid aqui para ser consistente
meu_grid_final = {
    'n_estimators': [50, 100, 200, 400],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

In [7]:
from model_utils import (
    split_age_groups,
    run_random_forest_pipeline,
    get_model_predictions,
    display_group_report,
    export_visual_reports
)

from experiment_manager import ExperimentManager

from sklearn.model_selection import train_test_split

# 1. Inicia o gerente (Ele cria as pastas run_YYYYMMDD_HHMM automaticamente)
exp = ExperimentManager(experiment_name="RF_Final_Groups_v1")

# 2. Separar os dados
groups, group_stats = split_age_groups(df, AGE_GROUPS)

for name, group_df in groups.items():
    print(f"\n--- Processando Grupo: {name.upper()} ---")

    # 3. Preparação dos Dados
    X = group_df.drop(columns=['obito'], errors = 'ignore')
    y = group_df['obito']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # 4. Treino e Otimização (GridSearch via model_utils)
    best_model, best_params = run_random_forest_pipeline(X_train, y_train, name, param_grid=meu_grid_final)

    # 5. Predições
    y_pred, y_proba = get_model_predictions(best_model, X_test)

    # 6. EXIBIÇÃO E ARMAZENAMENTO (A parte que você perguntou)
    # A função printa o relatório na tela e guarda o DataFrame no dicionário
    report_data = display_group_report(y_test, y_pred, y_proba, name)
    all_reports[name] = report_data

    # 7. Persistência (Salva o .pkl e o .json)
    exp.save_group(name, best_model, best_params, report_data)

    # 8. Exportação Silenciosa de Gráficos (Matriz de Confusão e Feature Importance)
    export_visual_reports(best_model, X_test, y_test, y_pred, name, save_path=exp.reports_path)



print("\n✅ Todos os grupos foram processados e armazenados em 'all_reports'!")

🚀 Gerente de Experimentos: RF_Final_Groups_v1
📂 Modelos em: models/run_20260417_1131

          📊 RELATÓRIO DE DIVISÃO POR FAIXA ETÁRIA           
🔹 IDOSO           | Range: [80, 120) | Amostras: 169663 | Óbitos: 42243 ( 24.9%)


--- Processando Grupo: IDOSO ---

🔍 Iniciando Grid Search: idoso
Fitting 3 folds for each of 144 candidates, totalling 432 fits

📊 RELATÓRIO DE CLASSIFICAÇÃO: IDOSO
              precision    recall  f1-score   support

           0       0.88      0.70      0.78     25484
           1       0.44      0.71      0.54      8449

    accuracy                           0.70     33933
   macro avg       0.66      0.70      0.66     33933
weighted avg       0.77      0.70      0.72     33933


✅ Todos os grupos foram processados e armazenados em 'all_reports'!


In [10]:

exp.finalize(metadata={
    "grid_search": meu_grid_final,
    "dataset_stats": group_stats,
    "global_samples": len(df)
})

🏁 Relatório Final Gerado: models/run_20260417_1131/full_experiment_results.json


In [11]:
import sys
import shutil

# Verifica se o ambiente é o Google Colab
if 'google.colab' in sys.modules:
    from google.colab import files
    print("☁️ Ambiente Colab detectado. Preparando downloads...")

    # 1. Compactar e Baixar MODELOS e METADADOS (exp.base_path)
    model_zip = f"{exp.run_id}_models"
    shutil.make_archive(model_zip, 'zip', exp.base_path)
    files.download(f"{model_zip}.zip")
    print(f"✅ Download do ZIP de modelos ({model_zip}.zip) iniciado.")

    # 2. Compactar e Baixar RELATÓRIOS E GRÁFICOS (exp.reports_path)
    report_zip = f"{exp.run_id}_reports"
    shutil.make_archive(report_zip, 'zip', exp.reports_path)
    files.download(f"{report_zip}.zip")
    print(f"✅ Download do ZIP de gráficos ({report_zip}.zip) iniciado.")

else:
    print("💻 Ambiente Local detectado. Os arquivos já estão salvos em:")
    print(f"📂 Modelos: {exp.base_path}")
    print(f"📂 Gráficos: {exp.reports_path}")

☁️ Ambiente Colab detectado. Preparando downloads...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download do ZIP de modelos (run_20260417_1131_models.zip) iniciado.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download do ZIP de gráficos (run_20260417_1131_reports.zip) iniciado.
